<div align="center">
  <img src="https://raw.githubusercontent.com/harishs-io/lateral_flowimmunoassay_quant/main/assets/pipeline_workflow.svg" width="850">
  <h1>🧬 Automated LFIA Signal Quantification</h1>
  <h3>Publication-grade bioanalytical pipeline using YOLOv8</h3>
</div>

---
**Instructions:**
1. Run the **Environment Setup** cell to load the pipeline.
2. Run the **Upload & Analyze** cell to insert your image and extract the signal.
3. Run the **Export** cell to download your data.

In [ ]:
#@title ⚙️ 1. Environment Setup { display-mode: "form" }
#@markdown Run this cell once to configure the workspace. *(Installation logs are hidden for a cleaner interface).* 

%%capture
!git clone https://github.com/harishs-io/lateral_flowimmunoassay_quant.git
%cd lateral_flowimmunoassay_quant
!pip install ultralytics opencv-python matplotlib scipy pandas numpy

import os
import sys
sys.path.append('.')
from google.colab import files
from IPython.display import clear_output

clear_output()
print("✅ Pipeline environment successfully loaded! Move to Step 2.")

In [ ]:
#@title 🔬 2. Upload Image & Run Pipeline { display-mode: "form" }
#@markdown Adjust the detection parameters below, then click play. You will be prompted to select your image file.

yolo_confidence = 0.75 #@param {type:"slider", min:0.1, max:1.0, step:0.05}
baseline_correction = "Dynamic" #@param ["Dynamic", "Static", "None"]

print("👇 Click 'Choose Files' below to insert your assay image:")
uploaded = files.upload()

if uploaded:
    image_path = list(uploaded.keys())[0]
    clear_output()
    print(f"✅ Successfully uploaded: {image_path}")
    print("⏳ Processing image through YOLOv8 and extracting chromatogram...")
    
    try:
        from src.lfia_analyzer import LFIAAnalyzer
        
        analyzer = LFIAAnalyzer(weights='weights/best.pt', conf=yolo_confidence)
        results = analyzer.process_image(image_path, baseline=baseline_correction)
        
        clear_output()
        print("✅ Pipeline complete! Target localized and integrated.")
        results.plot_chromatogram()
        
    except Exception as e:
        print(f"\n⚠️ Pipeline Error: {e}")
        print("Ensure model weights are in the 'weights/' directory.")
else:
    print("⚠️ No file uploaded. Please run the cell again.")

In [ ]:
#@title 📊 3. View Calibration & Export Data { display-mode: "form" }
#@markdown Generates the 4PL regression curve and downloads the CSV report.

try:
    results.show_calibration_curve()
    results.export_csv('quantification_report.csv')
    
    print("✅ Results packaged. Triggering download for 'quantification_report.csv'...")
    files.download('quantification_report.csv')
except NameError:
    print("⚠️ Error: Please run Step 2 first to generate the results.")